Load logic for `adwm_wh.gold.factsales`.

Business grain:

* One row per sales order line
* Business key is `SalesOrderNumber` + `SalesOrderDetailID`
* The merge matches on that order-line grain to support incremental upserts without duplicating facts

Source tables:

* `adwm_wh.silver.salesorderheader`
* `adwm_wh.silver.salesorderdetail`
* `adwm_wh.gold.dimdate`
* `adwm_wh.gold.dimcustomer`
* `adwm_wh.gold.dimproduct`
* `adwm_wh.gold.dimemployee`

Incremental load behavior:

* The load accepts optional parameter `:load_from_ts`
* If the parameter is blank, the notebook derives the latest processed source change timestamp from rows already loaded in the target
* Existing fact rows are updated only when tracked source values change, using a row-level hash comparison
* Dimension joins first reduce each dimension to the latest row per business key to avoid row multiplication

Notebook contents:

* Incremental merge logic for the fact table
* Validation checks for row counts and business grain
* Validation checks for resolved surrogate keys
* Validation checks for reconciled measures

In [0]:
%sql
WITH load_control AS (
  SELECT COALESCE(
    TRY_CAST(NULLIF(:load_from_ts, '') AS TIMESTAMP),
    (
      SELECT COALESCE(
        MAX(
          GREATEST(
            COALESCE(soh.ModifiedDate, TIMESTAMP '1900-01-01 00:00:00'),
            COALESCE(sod.ModifiedDate, TIMESTAMP '1900-01-01 00:00:00')
          )
        ),
        TIMESTAMP '1900-01-01 00:00:00'
      )
      FROM adwm_wh.gold.factsales AS f
      INNER JOIN adwm_wh.silver.salesorderheader AS soh
        ON soh.SalesOrderID = f.SalesOrderID
      INNER JOIN adwm_wh.silver.salesorderdetail AS sod
        ON sod.SalesOrderID = f.SalesOrderID
       AND sod.SalesOrderDetailID = f.SalesOrderDetailID
    )
  ) AS load_from_ts
),
dim_customer_latest AS (
  SELECT CustomerID, CustomerKey
  FROM (
    SELECT
      CustomerID,
      CustomerKey,
      ROW_NUMBER() OVER (
        PARTITION BY CustomerID
        ORDER BY modified_date DESC, CustomerKey DESC
      ) AS rn
    FROM adwm_wh.gold.dimcustomer
  ) c
  WHERE rn = 1
),
dim_product_latest AS (
  SELECT ProductID, ProductKey, StandardCost
  FROM (
    SELECT
      ProductID,
      ProductKey,
      StandardCost,
      ROW_NUMBER() OVER (
        PARTITION BY ProductID
        ORDER BY modified_date DESC, ProductKey DESC
      ) AS rn
    FROM adwm_wh.gold.dimproduct
  ) p
  WHERE rn = 1
),
dim_employee_latest AS (
  SELECT EmployeeID, EmployeeKey
  FROM (
    SELECT
      EmployeeID,
      EmployeeKey,
      ROW_NUMBER() OVER (
        PARTITION BY EmployeeID
        ORDER BY COALESCE(IsActive, false) DESC, modified_date DESC, EmployeeKey DESC
      ) AS rn
    FROM adwm_wh.gold.dimemployee
  ) e
  WHERE rn = 1
),
source_sales AS (
  SELECT
    COALESCE(od.DateKey, -1) AS OrderDateKey,
    COALESCE(sd.DateKey, -1) AS ShipDateKey,
    COALESCE(dc.CustomerKey, -1) AS CustomerKey,
    COALESCE(de.EmployeeKey, -1) AS EmployeeKey,
    COALESCE(dp.ProductKey, -1) AS ProductKey,
    soh.SalesOrderNumber,
    sod.SalesOrderDetailID AS SalesOrderLineNumber,
    sod.OrderQty AS OrderQuantity,
    CAST(sod.UnitPrice AS DECIMAL(19,4)) AS UnitPrice,
    CAST(COALESCE(dp.StandardCost, 0) AS DECIMAL(19,4)) AS UnitCost,
    CAST(sod.UnitPrice * sod.UnitPriceDiscount * sod.OrderQty AS DECIMAL(19,4)) AS DiscountAmount,
    CAST(sod.LineTotal AS DECIMAL(19,4)) AS SalesAmount,
    CAST(COALESCE(dp.StandardCost, 0) * sod.OrderQty AS DECIMAL(19,4)) AS TotalCost,
    soh.SalesOrderID,
    sod.SalesOrderDetailID,
    soh.CustomerID,
    soh.SalesPersonID,
    sod.ProductID,
    sha2(
      concat_ws(
        '||',
        CAST(COALESCE(od.DateKey, -1) AS STRING),
        CAST(COALESCE(sd.DateKey, -1) AS STRING),
        CAST(COALESCE(dc.CustomerKey, -1) AS STRING),
        CAST(COALESCE(de.EmployeeKey, -1) AS STRING),
        CAST(COALESCE(dp.ProductKey, -1) AS STRING),
        COALESCE(soh.SalesOrderNumber, ''),
        CAST(COALESCE(sod.SalesOrderDetailID, -1) AS STRING),
        CAST(COALESCE(sod.OrderQty, 0) AS STRING),
        CAST(CAST(sod.UnitPrice AS DECIMAL(19,4)) AS STRING),
        CAST(CAST(COALESCE(dp.StandardCost, 0) AS DECIMAL(19,4)) AS STRING),
        CAST(CAST(sod.UnitPrice * sod.UnitPriceDiscount * sod.OrderQty AS DECIMAL(19,4)) AS STRING),
        CAST(CAST(sod.LineTotal AS DECIMAL(19,4)) AS STRING),
        CAST(CAST(COALESCE(dp.StandardCost, 0) * sod.OrderQty AS DECIMAL(19,4)) AS STRING),
        CAST(COALESCE(soh.SalesOrderID, -1) AS STRING),
        CAST(COALESCE(sod.SalesOrderDetailID, -1) AS STRING),
        CAST(COALESCE(soh.CustomerID, -1) AS STRING),
        CAST(COALESCE(soh.SalesPersonID, -1) AS STRING),
        CAST(COALESCE(sod.ProductID, -1) AS STRING)
      ),
      256
    ) AS RowHash,
    current_timestamp() AS FactLoadedAt
  FROM adwm_wh.silver.salesorderheader AS soh
  INNER JOIN adwm_wh.silver.salesorderdetail AS sod
    ON sod.SalesOrderID = soh.SalesOrderID
  LEFT JOIN adwm_wh.gold.dimdate AS od
    ON od.FullDate = CAST(soh.OrderDate AS DATE)
  LEFT JOIN adwm_wh.gold.dimdate AS sd
    ON sd.FullDate = CAST(soh.ShipDate AS DATE)
  LEFT JOIN dim_customer_latest AS dc
    ON dc.CustomerID = soh.CustomerID
  LEFT JOIN dim_product_latest AS dp
    ON dp.ProductID = sod.ProductID
  LEFT JOIN dim_employee_latest AS de
    ON de.EmployeeID = soh.SalesPersonID
  CROSS JOIN load_control AS lc
  WHERE GREATEST(
    COALESCE(soh.ModifiedDate, TIMESTAMP '1900-01-01 00:00:00'),
    COALESCE(sod.ModifiedDate, TIMESTAMP '1900-01-01 00:00:00')
  ) >= lc.load_from_ts
)
MERGE INTO adwm_wh.gold.factsales AS tgt
USING source_sales AS src
ON tgt.SalesOrderNumber = src.SalesOrderNumber
AND tgt.SalesOrderDetailID = src.SalesOrderDetailID
WHEN MATCHED AND sha2(
  concat_ws(
    '||',
    CAST(COALESCE(tgt.OrderDateKey, -1) AS STRING),
    CAST(COALESCE(tgt.ShipDateKey, -1) AS STRING),
    CAST(COALESCE(tgt.CustomerKey, -1) AS STRING),
    CAST(COALESCE(tgt.EmployeeKey, -1) AS STRING),
    CAST(COALESCE(tgt.ProductKey, -1) AS STRING),
    COALESCE(tgt.SalesOrderNumber, ''),
    CAST(COALESCE(tgt.SalesOrderDetailID, -1) AS STRING),
    CAST(COALESCE(tgt.OrderQuantity, 0) AS STRING),
    CAST(CAST(tgt.UnitPrice AS DECIMAL(19,4)) AS STRING),
    CAST(CAST(COALESCE(tgt.UnitCost, 0) AS DECIMAL(19,4)) AS STRING),
    CAST(CAST(COALESCE(tgt.DiscountAmount, 0) AS DECIMAL(19,4)) AS STRING),
    CAST(CAST(COALESCE(tgt.SalesAmount, 0) AS DECIMAL(19,4)) AS STRING),
    CAST(CAST(COALESCE(tgt.TotalCost, 0) AS DECIMAL(19,4)) AS STRING),
    CAST(COALESCE(tgt.SalesOrderID, -1) AS STRING),
    CAST(COALESCE(tgt.SalesOrderDetailID, -1) AS STRING),
    CAST(COALESCE(tgt.CustomerID, -1) AS STRING),
    CAST(COALESCE(tgt.SalesPersonID, -1) AS STRING),
    CAST(COALESCE(tgt.ProductID, -1) AS STRING)
  ),
  256
) <> src.RowHash THEN UPDATE SET
  tgt.OrderDateKey = src.OrderDateKey,
  tgt.ShipDateKey = src.ShipDateKey,
  tgt.CustomerKey = src.CustomerKey,
  tgt.EmployeeKey = src.EmployeeKey,
  tgt.ProductKey = src.ProductKey,
  tgt.SalesOrderNumber = src.SalesOrderNumber,
  tgt.SalesOrderLineNumber = src.SalesOrderLineNumber,
  tgt.OrderQuantity = src.OrderQuantity,
  tgt.UnitPrice = src.UnitPrice,
  tgt.UnitCost = src.UnitCost,
  tgt.DiscountAmount = src.DiscountAmount,
  tgt.SalesAmount = src.SalesAmount,
  tgt.TotalCost = src.TotalCost,
  tgt.SalesOrderID = src.SalesOrderID,
  tgt.SalesOrderDetailID = src.SalesOrderDetailID,
  tgt.CustomerID = src.CustomerID,
  tgt.SalesPersonID = src.SalesPersonID,
  tgt.ProductID = src.ProductID,
  tgt.FactLoadedAt = src.FactLoadedAt
WHEN NOT MATCHED THEN INSERT (
  OrderDateKey,
  ShipDateKey,
  CustomerKey,
  EmployeeKey,
  ProductKey,
  SalesOrderNumber,
  SalesOrderLineNumber,
  OrderQuantity,
  UnitPrice,
  UnitCost,
  DiscountAmount,
  SalesAmount,
  TotalCost,
  SalesOrderID,
  SalesOrderDetailID,
  CustomerID,
  SalesPersonID,
  ProductID,
  FactLoadedAt
)
VALUES (
  src.OrderDateKey,
  src.ShipDateKey,
  src.CustomerKey,
  src.EmployeeKey,
  src.ProductKey,
  src.SalesOrderNumber,
  src.SalesOrderLineNumber,
  src.OrderQuantity,
  src.UnitPrice,
  src.UnitCost,
  src.DiscountAmount,
  src.SalesAmount,
  src.TotalCost,
  src.SalesOrderID,
  src.SalesOrderDetailID,
  src.CustomerID,
  src.SalesPersonID,
  src.ProductID,
  src.FactLoadedAt
);

Validation checks for `adwm_wh.gold.factsales`.

Coverage:

* Source-to-target row count reconciliation
* Duplicate detection at `SalesOrderNumber` + `SalesOrderDetailID` grain
* Invalid or unresolved date and dimension key counts
* Measure reconciliation for quantity and sales amount

Expected outcome after a successful load:

* Source and target row counts match
* Duplicate grain count is zero
* Invalid key counts are zero
* Aggregate measure differences reconcile to zero

In [0]:
%sql
WITH source_counts AS (
  SELECT
    COUNT(*) AS source_row_count,
    COUNT(DISTINCT CONCAT(soh.SalesOrderNumber, ':', CAST(sod.SalesOrderDetailID AS STRING))) AS source_distinct_grain
  FROM adwm_wh.silver.salesorderheader AS soh
  INNER JOIN adwm_wh.silver.salesorderdetail AS sod
    ON sod.SalesOrderID = soh.SalesOrderID
),
target_counts AS (
  SELECT
    COUNT(*) AS target_row_count,
    COUNT(DISTINCT CONCAT(SalesOrderNumber, ':', CAST(SalesOrderDetailID AS STRING))) AS target_distinct_grain
  FROM adwm_wh.gold.factsales
),
duplicate_check AS (
  SELECT COUNT(*) AS duplicate_grain_groups
  FROM (
    SELECT SalesOrderNumber, SalesOrderDetailID
    FROM adwm_wh.gold.factsales
    GROUP BY SalesOrderNumber, SalesOrderDetailID
    HAVING COUNT(*) > 1
  ) d
)
SELECT
  s.source_row_count,
  s.source_distinct_grain,
  t.target_row_count,
  t.target_distinct_grain,
  d.duplicate_grain_groups,
  t.target_row_count - s.source_row_count AS row_count_difference
FROM source_counts s
CROSS JOIN target_counts t
CROSS JOIN duplicate_check d;

In [0]:
%sql
SELECT
  SUM(CASE WHEN OrderDateKey IS NULL OR OrderDateKey = -1 THEN 1 ELSE 0 END) AS invalid_order_date_keys,
  SUM(CASE WHEN ShipDateKey IS NULL OR ShipDateKey = -1 THEN 1 ELSE 0 END) AS invalid_ship_date_keys,
  SUM(CASE WHEN CustomerKey IS NULL OR CustomerKey = -1 THEN 1 ELSE 0 END) AS invalid_customer_keys,
  SUM(CASE WHEN EmployeeKey IS NULL OR EmployeeKey = -1 THEN 1 ELSE 0 END) AS invalid_employee_keys,
  SUM(CASE WHEN ProductKey IS NULL OR ProductKey = -1 THEN 1 ELSE 0 END) AS invalid_product_keys,
  COUNT(*) AS total_rows
FROM adwm_wh.gold.factsales;

In [0]:
%sql
WITH source_measures AS (
  SELECT
    COUNT(*) AS source_rows,
    SUM(sod.OrderQty) AS source_order_qty,
    CAST(SUM(sod.LineTotal) AS DECIMAL(19,4)) AS source_sales_amount
  FROM adwm_wh.silver.salesorderheader AS soh
  INNER JOIN adwm_wh.silver.salesorderdetail AS sod
    ON sod.SalesOrderID = soh.SalesOrderID
),
target_measures AS (
  SELECT
    COUNT(*) AS target_rows,
    SUM(OrderQuantity) AS target_order_qty,
    CAST(SUM(SalesAmount) AS DECIMAL(19,4)) AS target_sales_amount
  FROM adwm_wh.gold.factsales
)
SELECT
  s.source_rows,
  t.target_rows,
  s.source_order_qty,
  t.target_order_qty,
  s.source_sales_amount,
  t.target_sales_amount,
  CAST(t.target_sales_amount - s.source_sales_amount AS DECIMAL(19,4)) AS sales_amount_difference
FROM source_measures s
CROSS JOIN target_measures t;